In [1]:
import numpy as np
from LanzaModels import TVL1_1D
from ADMMsRustici import MyBackTrackingSolver
from signalClass import *
import time

In [2]:
np.random.seed(24102000)
n = 128

construct blur matrix

In [3]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [4]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 15)
RndSignal.generate_GG_realization(0, sigma, 1)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [5]:
mu = 0.5
VarModel = TVL1_1D.TVL1_1DClass(A, xCorrupted, mu)

Lfid = mu * np.max(np.linalg.svdvals(A)) * np.sqrt(n)
Lreg = np.sqrt(n)
Lphi = np.sqrt(Lfid**2 + Lreg**2)

Now, we need to initialize and define the solver

In [6]:
#begin solver construction
np.random.seed(24102002)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = MyBackTrackingSolver.MyBacktrackingSolverClass(VarModel, xk, yk, lk, betak, Lphi)

#end solver construction

In [7]:
iters = 10

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [8]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = betak_1 * np.linalg.norm(VarModel.P.T @ (VarModel.Q @ (yk - yk_1)) )

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    ImgHistory[iter] = VarModel(xk_1, yk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResidue) <= 1e-9):
        print(iter)
        break


1 / 10
iter: 5
ImgErr: 5.938438354535492
DualResNorm: 0.40017270890122203
ImgErr: 4.765090567062094
DualResNorm: 0.3399483314605282
ImgErr: 3.4713022422316726
DualResNorm: 0.25151117762467884
ImgErr: 2.6472702935879853
DualResNorm: 0.1938991783757596
ImgErr: 2.0241781177524336
DualResNorm: 0.14947086172385426
ImgErr: 1.5561478744903678
DualResNorm: 0.1156071452824894
ImgErr: 1.2044340152785635
DualResNorm: 0.08988398870451896
ImgErr: 0.9315533564583106
DualResNorm: 0.06976040075580066
ImgErr: 0.676182888115311
DualResNorm: 0.050787647875475586
ImgErr: 0.4916235757529754
DualResNorm: 0.037004203346173416
ImgErr: 0.3464763188909727
DualResNorm: 0.026121124989765456
ImgErr: 0.2215806843212558
DualResNorm: 0.016727377799920215
ImgErr: 0.142391964300965
DualResNorm: 0.010758343075221085
ImgErr: 0.09211302431231087
DualResNorm: 0.006963215388911471
ImgErr: 0.05970127393893581
DualResNorm: 0.004514654487902491
ImgErr: 0.03918752498910492
DualResNorm: 0.0029640374140427242
ImgErr: 0.0252974935

In [9]:
np.savez_compressed(
    "./MyADMMTVL1-Laplace.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)